# Lab 11 — QAOA: Paisaje del Coste

Analizamos el paisaje de la función de coste QAOA para el problema MaxCut.
Estudiamos cómo varía con el número de capas p y el tamaño del grafo.

**Objetivo**: entender la geometría del espacio de parámetros (γ, β) y sus implicaciones para la optimización.

## 1. Configuración: MaxCut en un triángulo

Usamos un grafo triangular (3 nodos, 3 aristas) para visualizar el paisaje 2D completo con p=1.

El MaxCut del triángulo es 2 aristas (una partición de 1 vs 2 nodos).
Hay 6 soluciones equivalentes: `001`, `010`, `100`, `011`, `101`, `110`.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from scipy.optimize import minimize
from qiskit import QuantumCircuit
from qiskit.circuit import ParameterVector
from qiskit.quantum_info import SparsePauliOp
from qiskit.primitives import StatevectorEstimator, StatevectorSampler

# Grafo triángulo
edges_tri = [(0, 1), (1, 2), (0, 2)]
n = 3

def build_cost_ham(edges, n_qubits):
    pauli_list = []
    for (i, j) in edges:
        ops = ['I'] * n_qubits
        ops[i] = 'Z'; ops[j] = 'Z'
        pauli_list.append((''.join(reversed(ops)), -0.5))
    return SparsePauliOp.from_list(pauli_list)

def build_qaoa(edges, n_qubits, p):
    gammas = ParameterVector('γ', p)
    betas = ParameterVector('β', p)
    qc = QuantumCircuit(n_qubits)
    qc.h(range(n_qubits))
    for layer in range(p):
        for (i, j) in edges:
            qc.cx(i, j)
            qc.rz(2 * gammas[layer], j)
            qc.cx(i, j)
        for q in range(n_qubits):
            qc.rx(2 * betas[layer], q)
    return qc

H_tri = build_cost_ham(edges_tri, n)
E_exact = np.linalg.eigvalsh(H_tri.to_matrix())
print(f'MaxCut triángulo: corte max = {-E_exact[0]:.2f} aristas')

## 2. Paisaje 2D para p=1

Con p=1, los parámetros son (γ₀, β₀) ∈ [0, π] × [0, π/2] (simetría del paisaje).
El paisaje tiene estructura periódica y múltiples mínimos.

In [ ]:
estimator = StatevectorEstimator()
qaoa_p1 = build_qaoa(edges_tri, n, p=1)

n_pts = 40
gammas = np.linspace(0, np.pi, n_pts)
betas = np.linspace(0, np.pi / 2, n_pts)
Z = np.zeros((n_pts, n_pts))

for i, g in enumerate(gammas):
    for j, b in enumerate(betas):
        job = estimator.run([(qaoa_p1, H_tri, [g, b])])
        Z[i, j] = float(job.result()[0].data.evs)

fig, ax = plt.subplots(figsize=(7, 6))
c = ax.contourf(gammas / np.pi, betas / np.pi, Z.T, levels=30, cmap='RdBu')
plt.colorbar(c, ax=ax, label='⟨H_cost⟩')
min_idx = np.unravel_index(Z.argmin(), Z.shape)
ax.plot(gammas[min_idx[0]] / np.pi, betas[min_idx[1]] / np.pi,
        'r*', markersize=15, label=f'Mínimo: ⟨H⟩={Z.min():.3f}')
ax.set_xlabel('γ / π', fontsize=12)
ax.set_ylabel('β / π', fontsize=12)
ax.set_title('Paisaje de coste QAOA p=1 — MaxCut triángulo', fontsize=13)
ax.legend(fontsize=10)
plt.tight_layout()
plt.show()

print(f'Mínimo del paisaje: ⟨H⟩ = {Z.min():.4f}')
print(f'E₀ exacto:          {E_exact[0]:.4f}')

## 3. Efecto del número de capas p

A mayor p, QAOA puede alcanzar valores más bajos de energía (mejor aproximación al óptimo).
Pero el espacio de parámetros crece y la optimización se vuelve más difícil.

In [ ]:
cortes_vs_p = {}

for p in [1, 2, 3, 4]:
    qaoa_p = build_qaoa(edges_tri, n, p=p)
    best = np.inf
    for seed in range(5):
        np.random.seed(seed)
        x0 = np.random.uniform(0, np.pi, qaoa_p.num_parameters)
        def cost(params):
            return float(estimator.run([(qaoa_p, H_tri, params)]).result()[0].data.evs)
        res = minimize(cost, x0, method='COBYLA', options={'maxiter': 500})
        if res.fun < best:
            best = res.fun
    cortes_vs_p[p] = -best

fig, ax = plt.subplots(figsize=(7, 4))
ps = list(cortes_vs_p.keys())
cortes = list(cortes_vs_p.values())
ax.plot(ps, cortes, 'bo-', markersize=8, linewidth=2, label='QAOA')
ax.axhline(2.0, color='red', linestyle='--', label='MaxCut óptimo = 2')
ax.set_xlabel('Capas p', fontsize=12)
ax.set_ylabel('Corte aproximado', fontsize=12)
ax.set_title('QAOA vs p — MaxCut triángulo', fontsize=13)
ax.set_ylim(1.5, 2.2)
ax.legend()
ax.grid(alpha=0.3)
plt.tight_layout()
plt.show()

## 4. Concentración del paisaje: grafos grandes

Un resultado teórico importante: para grafos aleatorios grandes, el paisaje QAOA con p fijo
se **concentra** — los parámetros óptimos son casi los mismos para diferentes instancias.
Esto sugiere que se pueden preentrenar parámetros QAOA.

In [ ]:
# Comparar paisajes para diferentes grafos de 4 nodos
grafos = {
    'Ciclo C4': [(0,1),(1,2),(2,3),(3,0)],
    'Camino P4': [(0,1),(1,2),(2,3)],
    'Estrella K1,3': [(0,1),(0,2),(0,3)],
}

print('Grafo        | p=1 corte | p=2 corte | MaxCut exacto')
print('-' * 55)
for nombre, edges in grafos.items():
    H = build_cost_ham(edges, 4)
    E0 = np.linalg.eigvalsh(H.to_matrix())[0]
    mc_exacto = -E0

    row = f'{nombre:<12} |'
    for p in [1, 2]:
        qaoa = build_qaoa(edges, 4, p)
        best = np.inf
        for seed in range(3):
            np.random.seed(seed)
            x0 = np.random.uniform(0, np.pi, qaoa.num_parameters)
            def cost(params, q=qaoa, h=H):
                return float(estimator.run([(q, h, params)]).result()[0].data.evs)
            res = minimize(cost, x0, method='COBYLA', options={'maxiter': 400})
            if res.fun < best: best = res.fun
        row += f' {-best:.3f}      |'
    print(row + f' {mc_exacto:.3f}')

## 5. Resumen

**Propiedades del paisaje QAOA**:
- Para p=1, el paisaje es periódico y tiene estructura analítica conocida
- A mayor p, más expresividad pero más mínimos locales
- Para grafos aleatorios d-regulares: parámetros óptimos son universales (concentración)
- La relación aproximación QAOA vs algoritmo clásico óptimo es un área activa de investigación

**Referencia teórica**: Farhi et al. (2014) QAOA original; Brandao et al. (2018) concentración del paisaje.

In [ ]:
# Ratio de aproximación vs óptimo
print('Grafo        | Ratio QAOA p=1 | Ratio QAOA p=2')
print('-' * 50)
for nombre, edges in grafos.items():
    H = build_cost_ham(edges, 4)
    E0 = np.linalg.eigvalsh(H.to_matrix())[0]
    mc_exacto = -E0
    ratios = []
    for p in [1, 2]:
        qaoa = build_qaoa(edges, 4, p)
        best = np.inf
        for seed in range(3):
            np.random.seed(seed)
            x0 = np.random.uniform(0, np.pi, qaoa.num_parameters)
            def cost(params, q=qaoa, h=H):
                return float(estimator.run([(q, h, params)]).result()[0].data.evs)
            res = minimize(cost, x0, method='COBYLA', options={'maxiter': 400})
            if res.fun < best: best = res.fun
        ratios.append(-best / mc_exacto if mc_exacto > 0 else 0)
    print(f'{nombre:<12} | {ratios[0]:.4f}          | {ratios[1]:.4f}')